# Out-Of-Sample (OOS) inference for virtual staining
## A model per marker for all shared markers (11 out of 14; 3 markers (dna1, dna2, and H3) used for normaliztion):
### Train: Jackson2020 dataset 
### Test: Danenberg2022 datasets 

In [1]:
!pip install pytorch_msssim
!pip install imagecodecs
!pip install torchmetrics
!pip install clean-fid
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 41.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [ ]:
import os
import tifffile
import tempfile
import pickle
import json
import imageio
import shutil
import numpy as np
import pandas as pd
import gc
import time
import glob
import random
from scipy.stats import pearsonr
from sklearn.decomposition import IncrementalPCA
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.parametrizations import weight_norm
from torch.utils.data import DataLoader, random_split, Dataset
from torch.amp import GradScaler, autocast
from torchvision import models
import skimage.transform
from pytorch_msssim import ms_ssim
from cleanfid import fid
import optuna
from optuna.trial import Trial

import matplotlib.pyplot as plt
params = {
    'legend.fontsize': 16,
    'figure.figsize': (16, 10),
    'axes.labelsize': 16,
    'axes.titlesize': 12,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'figure.titlesize': 30,
}
plt.rcParams.update(params)
plt.style.use('seaborn-v0_8-whitegrid')

In [3]:
INPUT_CACHE_ROOT = "/kaggle/input/your-notebook-output-files/cv_cache"  # from the “Add Data” you attached
WORK_CACHE_ROOT  = "/kaggle/working/cv_cache"

# restore previous cache if present
if os.path.exists(INPUT_CACHE_ROOT):
    os.makedirs(WORK_CACHE_ROOT, exist_ok=True)
    for src in glob.glob(os.path.join(INPUT_CACHE_ROOT, "**/*"), recursive=True):
        if os.path.isfile(src):
            dst = src.replace(INPUT_CACHE_ROOT, WORK_CACHE_ROOT, 1)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
    print(f"[init] Restored cache from {INPUT_CACHE_ROOT} -> {WORK_CACHE_ROOT}")
else:
    os.makedirs(WORK_CACHE_ROOT, exist_ok=True)
    print("No previous cache found; starting fresh.")

No previous cache found; starting fresh.


In [ ]:
class ResNet50Encoder(nn.Module):
    def __init__(self, in_channels=3, pretrained_conv1=False):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        if pretrained_conv1 and in_channels == 3:
            self.conv1 = backbone.conv1
            self.conv1.requires_grad_(True)
        else:
            self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        for param in backbone.parameters():
            param.requires_grad = False
            
        for param in self.layer2.parameters():
            param.requires_grad = True
            
        for param in self.layer3.parameters():
            param.requires_grad = True
            
        for param in self.layer4.parameters():
            param.requires_grad = True

    def forward(self, x):
        f0 = self.conv1(x)
        f0 = self.bn1(f0)
        f0 = self.relu(f0)
        f0_pool = self.maxpool(f0)
        f1 = self.layer1(f0_pool)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return f0, f1, f2, f3, f4

class UNetDecoder(nn.Module):
    def __init__(self, out_channels=1, p=0.2):
        super().__init__()
        # Stage f4(2048) -> f3 scale (target 1024 ch)
        self.conv4_1 = nn.Sequential(
            nn.Conv2d(2048 + 1024, 1024, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p),
        )
        self.conv4_2 = nn.Sequential(
            nn.Conv2d(1024, 1024, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
        )

        # Stage -> f2 scale (target 512 ch)
        self.conv3_1 = nn.Sequential(
            nn.Conv2d(1024 + 512, 512, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p),
        )
        self.conv3_2 = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )

        # Stage -> f1 scale (target 256 ch)
        self.conv2_1 = nn.Sequential(
            nn.Conv2d(512 + 256, 256, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p),
        )
        self.conv2_2 = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        # Stage -> combine (f1 + f0) at f1 scale (target 64 ch)
        self.conv1_1 = nn.Sequential(
            nn.Conv2d(256 + 256 + 64, 64, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p),
        )
        self.conv1_2 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # Final upsample 
        self.final_up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

        # Light head
        self.up0 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.final = weight_norm(nn.Conv2d(32, out_channels, 1))

    def forward(self, skips):
        f0, f1, f2, f3, f4 = skips 

        # ---- up4: f4 -> f3 scale ----
        x = F.interpolate(f4, scale_factor=2, mode='bilinear', align_corners=False)
        if x.shape[-2:] != f3.shape[-2:]:
            f3 = F.interpolate(f3, size=x.shape[-2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, f3], dim=1)
        x = self.conv4_1(x)
        x = self.conv4_2(x)

        # ---- up3: -> f2 scale ----
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        if x.shape[-2:] != f2.shape[-2:]:
            f2 = F.interpolate(f2, size=x.shape[-2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, f2], dim=1)
        x = self.conv3_1(x)
        x = self.conv3_2(x)

        # ---- up2: -> f1 scale ----
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        if x.shape[-2:] != f1.shape[-2:]:
            f1 = F.interpolate(f1, size=x.shape[-2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, f1], dim=1)
        x = self.conv2_1(x)
        x = self.conv2_2(x)

        # ---- up1: combine f1 & f0 at f1 scale ----
        if f0.shape[-2:] != f1.shape[-2:]:
            f0 = F.interpolate(f0, size=f1.shape[-2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, f1, f0], dim=1)
        x = self.conv1_1(x)
        x = self.conv1_2(x)

        # ---- final upsample + head ----
        x = self.final_up(x) 
        x = self.up0(x)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False) 
        x = torch.sigmoid(self.final(x))
        return x

class ResNetUNet(nn.Module):
    def __init__(self, in_channels=3, pretrained_conv1=False):
        super().__init__()
        self.encoder = ResNet50Encoder(in_channels=in_channels, pretrained_conv1=pretrained_conv1)
        self.decoder = UNetDecoder()

    def forward(self, x):
        skips = self.encoder(x)
        return self.decoder(skips)

class CustomDataset(Dataset):
    def __init__(self, file_list, target_idx, control_markers_indices,
                 regression_models, pca, pca_mean, pca_std, input_channels,
                 n_components=3, cofactor=5.0, chunk_size=100_000, index_map=None):
        self.file_list = file_list
        self.target_idx = target_idx
        self.control_markers_indices = control_markers_indices
        self.regression_models = regression_models
        self.pca = pca
        self.pca_mean = pca_mean
        self.pca_std = pca_std
        self.input_channels = input_channels
        self.n_components = n_components
        self.cofactor = cofactor
        self.chunk_size = chunk_size
        self.index_map = index_map

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_path = self.file_list[idx]
        x_tensor, y_tensor, _ = preprocess_image(
            file_path,
            self.target_idx,
            self.control_markers_indices,
            self.regression_models,
            self.pca,
            self.pca_mean,
            self.pca_std,
            self.input_channels,
            self.n_components,
            self.cofactor,
            self.chunk_size,
            index_map=self.index_map
        )
        H, W = x_tensor.shape[1], x_tensor.shape[2]  
        return x_tensor, y_tensor, (H, W)

In [ ]:
def to_hwc(img):
    if img.ndim == 2:
        img = img[..., None]
    if img.shape[0] < img.shape[1] and img.shape[0] < img.shape[2]:
        img = np.transpose(img, (1, 2, 0))
    return img.astype(np.float32, copy=False)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def fit_preprocessing(
    train_files,
    target_idx,
    control_markers_indices,
    shared_marker_indices=None,
    n_components=3,
    batch_size=100_000,
    cofactor=5.0,
    eps=1e-8,
    use_pca=True):
    """
    Fit channel-wise regressions and (optionally) incremental PCA on residuals.

    use_pca=True  → regression → arcsinh → PCA (n_components) → z-score PCA components
    use_pca=False → regression → arcsinh → z-score each channel directly (no PCA)

    Returns: (regression_models, pca, pca_mean, pca_std, input_channels)
    When use_pca=False, pca=None and pca_mean/pca_std are per-channel stats (shape K).
    """
    img0 = to_hwc(tifffile.imread(train_files[0]))
    H, W, C = img0.shape

    if shared_marker_indices is not None:
        input_channels = [c for c in shared_marker_indices
                          if c != target_idx and c not in control_markers_indices]
    else:
        input_channels = [c for c in range(C)
                          if c != target_idx and c not in control_markers_indices]

    K = len(input_channels)
    del img0; gc.collect()

    if not input_channels:
        print(f"Warning: no inputs for target {target_idx} with controls {control_markers_indices}.")
        return None, None, None, None, input_channels

    # Pass 1: fit SGDRegressor per input channel (streaming)
    regression_models = {j: SGDRegressor(max_iter=1000, tol=1e-3) for j in input_channels}
    first_pass = {j: True for j in input_channels}

    for c, f in enumerate(train_files):
        if c % max(1, len(train_files) // 2) == 0 or c == len(train_files) - 1:
            print(f"[prepro] regression pass: {c+1} / {len(train_files)}")
        img = to_hwc(tifffile.imread(f))
        Xc = img[..., control_markers_indices].reshape(-1, len(control_markers_indices))
        for j in input_channels:
            y = img[..., j].reshape(-1)
            regression_models[j].partial_fit(Xc, y)
            first_pass[j] = False
        del img, Xc, y; gc.collect()

    if use_pca:
        # Pass 2: fit IncrementalPCA on arcsinh residuals
        if n_components > K:
            print(f"n_components {n_components} > residual dim {K}; using {K}.")
            n_components = K
        pca = IncrementalPCA(n_components=n_components)

        for c, f in enumerate(train_files):
            if c % max(1, len(train_files) // 2) == 0 or c == len(train_files) - 1:
                print(f"[prepro] PCA fit: {c+1} / {len(train_files)}")
            img = to_hwc(tifffile.imread(f))
            Xc = img[..., control_markers_indices].reshape(-1, len(control_markers_indices))
            R = np.empty((Xc.shape[0], K), dtype=np.float32)
            for k, j in enumerate(input_channels):
                y_pred = regression_models[j].predict(Xc)
                R[:, k] = img[..., j].reshape(-1) - y_pred.astype(np.float32, copy=False)
            R = np.arcsinh(R / cofactor).astype(np.float32)
            for s in range(0, R.shape[0], batch_size):
                pca.partial_fit(R[s:s + batch_size])
            del img, Xc, R; gc.collect()

        # Pass 3: compute mean/std of PCA-projected space
        total = 0
        sum_z   = np.zeros(n_components, dtype=np.float64)
        sumsq_z = np.zeros(n_components, dtype=np.float64)

        for c, f in enumerate(train_files):
            if c % max(1, len(train_files) // 2) == 0 or c == len(train_files) - 1:
                print(f"[prepro] PCA stats: {c+1} / {len(train_files)}")
            img = to_hwc(tifffile.imread(f))
            Xc = img[..., control_markers_indices].reshape(-1, len(control_markers_indices))
            R = np.empty((Xc.shape[0], K), dtype=np.float32)
            for k, j in enumerate(input_channels):
                y_pred = regression_models[j].predict(Xc)
                R[:, k] = img[..., j].reshape(-1) - y_pred.astype(np.float32, copy=False)
            R = np.arcsinh(R / cofactor).astype(np.float32)
            for s in range(0, R.shape[0], batch_size):
                Z = pca.transform(R[s:s + batch_size]).astype(np.float32)
                sum_z   += Z.sum(axis=0)
                sumsq_z += (Z * Z).sum(axis=0)
                total   += Z.shape[0]
            del img, Xc, R, Z; gc.collect()

        if total == 0:
            pca_mean = np.zeros(n_components, dtype=np.float32)
            pca_std  = np.ones(n_components,  dtype=np.float32) * eps
        else:
            pca_mean = (sum_z / total).astype(np.float32)
            var = (sumsq_z / total) - pca_mean.astype(np.float64) ** 2
            pca_std = (np.sqrt(np.maximum(var, 0.0)) + eps).astype(np.float32)

        if hasattr(pca, "explained_variance_ratio_") and pca.explained_variance_ratio_ is not None:
            pct = pca.explained_variance_ratio_[:n_components].sum() * 100.0
            print(f"[norm] PCA {n_components}D captures {pct:.1f}% variance")

    else:
        # No PCA: single pass to compute per-channel mean/std of arcsinh residuals
        pca = None
        total = 0
        sum_r   = np.zeros(K, dtype=np.float64)
        sumsq_r = np.zeros(K, dtype=np.float64)

        for c, f in enumerate(train_files):
            if c % max(1, len(train_files) // 2) == 0 or c == len(train_files) - 1:
                print(f"[prepro] channel stats (no PCA): {c+1} / {len(train_files)}")
            img = to_hwc(tifffile.imread(f))
            Xc = img[..., control_markers_indices].reshape(-1, len(control_markers_indices))
            R = np.empty((Xc.shape[0], K), dtype=np.float32)
            for k, j in enumerate(input_channels):
                y_pred = regression_models[j].predict(Xc)
                R[:, k] = img[..., j].reshape(-1) - y_pred.astype(np.float32, copy=False)
            R = np.arcsinh(R / cofactor).astype(np.float32)
            sum_r   += R.sum(axis=0).astype(np.float64)
            sumsq_r += (R * R).sum(axis=0).astype(np.float64)
            total   += R.shape[0]
            del img, Xc, R; gc.collect()

        if total == 0:
            pca_mean = np.zeros(K, dtype=np.float32)
            pca_std  = np.ones(K,  dtype=np.float32) * eps
        else:
            pca_mean = (sum_r / total).astype(np.float32)
            var = (sumsq_r / total) - pca_mean.astype(np.float64) ** 2
            pca_std = (np.sqrt(np.maximum(var, 0.0)) + eps).astype(np.float32)

        print(f"[norm] direct z-score across {K} channels (no PCA)")

    return regression_models, pca, pca_mean, pca_std, input_channels


def preprocess_image(
    file_path,
    target_idx,
    control_markers_indices,
    regression_models,
    pca,
    pca_mean,
    pca_std,
    input_channels,
    n_components=3,
    cofactor=5.0,
    chunk_size=100_000,
    index_map=None):
    """
    Preprocess a single IMC image:
      - normalize target (arcsinh + min-max)
      - compute regression residuals, arcsinh
      - if pca is not None: PCA project + z-score  (baseline / pretrained_conv1 conditions)
      - if pca is None:     z-score per channel     (no_pca condition)
    Returns (x_tensor [C,H,W], y_tensor [1,H,W], filename).
    """
    img = to_hwc(tifffile.imread(file_path))
    H, W, _ = img.shape
    filename = os.path.basename(file_path)

    def _r(i):
        return index_map[i] if index_map is not None else i

    # Target: arcsinh + min-max normalize to [0, 1]
    y_arc  = np.arcsinh(img[..., _r(target_idx)] / cofactor).astype(np.float32)
    y_min, y_max = y_arc.min(), y_arc.max()
    y_tensor = torch.from_numpy((y_arc - y_min) / (y_max - y_min + 1e-8)).unsqueeze(0).float()

    # Inputs: vectorized regression residuals
    n_ctrl = len(control_markers_indices)
    K      = len(input_channels)
    Xc  = img[..., [_r(i) for i in control_markers_indices]].reshape(-1, n_ctrl)
    Yin = np.stack([img[..., _r(j)].reshape(-1) for j in input_channels], axis=1)  # (N, K)

    B    = np.stack([regression_models[j].coef_      for j in input_channels], axis=-1).astype(np.float32)
    b    = np.array([regression_models[j].intercept_ for j in input_channels], dtype=np.float32).flatten()
    R    = np.arcsinh((Yin - (Xc @ B + b)) / cofactor).astype(np.float32)

    # Project and z-score
    N = R.shape[0]
    if pca is not None:
        Z = np.empty((N, n_components), dtype=np.float32)
        for s in range(0, N, chunk_size):
            Z[s:s + chunk_size] = pca.transform(R[s:s + chunk_size])
    else:
        Z = R  # (N, K)

    Z = ((Z - pca_mean) / (pca_std + 1e-8)).reshape(H, W, -1)
    x_tensor = torch.from_numpy(Z).permute(2, 0, 1).contiguous().float()

    return x_tensor, y_tensor, filename


def pad_collate(batch):
    """Pad variable-size images to the largest in the batch; return original sizes for cropping."""
    X_list, Y_list, original_sizes = zip(*batch)
    max_H = max(x.shape[1] for x in X_list)
    max_W = max(x.shape[2] for x in X_list)
    padded_X = torch.stack([F.pad(x, (0, max_W - x.shape[2], 0, max_H - x.shape[1])) for x in X_list])
    padded_Y = torch.stack([F.pad(y, (0, max_W - y.shape[2], 0, max_H - y.shape[1])) for y in Y_list])
    return padded_X, padded_Y, original_sizes


def train_model(model, train_loader, val_loader, device, title, config,
                epochs=60, huber_beta=1.0, grad_clip=1.0, use_amp=True,
                patience=20, save_path="best_model.pth", dataset=None):
    """
    Loss = alpha * Huber(ŷ, y) + (1 - alpha) * (1 - MS-SSIM(ŷ, y)).
    Huber on raw intensities; SSIM on the same [0,1]-normalized images.
    """
    model.to(device)
    lr, alpha = config['lr'], config['alpha']
    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)
    scaler = GradScaler(enabled=use_amp)
    epsilon = 1e-8

    train_losses, val_losses = [], []
    best_val, best_state, best_epoch = float("inf"), None, None
    no_improve = 0

    for epoch in range(epochs):
        # ---- train ----
        model.train()
        running = 0.0
        for x, y, original_sizes in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            B = x.shape[0]
            opt.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', enabled=use_amp):
                yhat = model(x)
                hub_sum = ssim_sum = 0.0
                for i in range(B):
                    h, w = original_sizes[i]
                    y_i, yhat_i = y[i:i+1, :, :h, :w], yhat[i:i+1, :, :h, :w]
                    hub_sum  += F.smooth_l1_loss(yhat_i, y_i, beta=huber_beta, reduction="mean")
                    ssim_sum += 1.0 - ms_ssim(yhat_i, y_i, data_range=1.0, win_size=7, weights=[0.4, 0.3, 0.3])
                loss = alpha * torch.as_tensor(hub_sum / B, device=device) \
                     + (1 - alpha) * torch.as_tensor(ssim_sum / B, device=device)
            scaler.scale(loss).backward()
            if grad_clip:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(opt); scaler.update()
            running += loss.item() * B

        train_loss = running / len(train_loader.dataset)
        train_losses.append(train_loss)

        # ---- validate ----
        model.eval()
        val_running, valid_batches = 0.0, 0
        with torch.no_grad(), autocast(device_type='cuda', enabled=use_amp):
            for x, y, original_sizes in val_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                B    = x.shape[0]
                yhat = model(x)
                hub_sum = ssim_sum = 0.0
                skip = False
                for i in range(B):
                    h, w     = original_sizes[i]
                    y_i      = y[i:i+1, :, :h, :w]
                    yhat_i   = yhat[i:i+1, :, :h, :w].clamp(0.0, 1.0)
                    if (y_i.max() - y_i.min()).abs() < epsilon or \
                       (yhat_i.max() - yhat_i.min()).abs() < epsilon:
                        skip = True; break
                    ssim_val = ms_ssim(yhat_i, y_i, data_range=1.0, win_size=7, weights=[0.4, 0.3, 0.3])
                    hub_val  = F.smooth_l1_loss(yhat_i, y_i, beta=huber_beta, reduction="mean")
                    if torch.isnan(ssim_val) or torch.isnan(hub_val):
                        skip = True; break
                    hub_sum  += hub_val
                    ssim_sum += 1.0 - ssim_val
                if skip:
                    continue
                val_loss_batch = (alpha * hub_sum + (1 - alpha) * ssim_sum) / B
                if not (torch.isnan(val_loss_batch) or torch.isinf(val_loss_batch)):
                    val_running += float(val_loss_batch) * B
                    valid_batches += B

        val_loss = val_running / valid_batches if valid_batches > 0 else float('nan')
        val_losses.append(val_loss)
        sched.step(val_loss)
        print(f"Epoch {epoch+1:>3d} | Train {train_loss:.4f} | Val {val_loss:.4f}")

        if val_loss + 1e-6 < best_val:
            best_val, best_epoch = val_loss, epoch + 1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"  ↳ New best val={val_loss:.4f} at epoch {best_epoch}")
            if save_path:
                torch.save(best_state, save_path)
        else:
            no_improve += 1
            print(f"  ↳ No improvement ({no_improve}/{patience})  LR={opt.param_groups[0]['lr']:.2e}")
            if patience and no_improve >= patience:
                print(f"-- Early stop at epoch {epoch+1}. Best: epoch {best_epoch}, val={best_val:.4f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"-- Restored weights from epoch {best_epoch}")

    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, marker='o', label='Train')
    plt.plot(val_losses,   marker='s', label='Val')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(f'Marker: {title}')
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

    return train_losses, val_losses


def visualize_prediction(model, dataset, device, indices=None, target_channel=0, name="comparison_grid"):
    """Show GT | Pred pairs.  Note: predictions are rescaled to GT statistics for display only."""
    model.to(device).eval()
    eps = 1e-8

    if indices is None:
        k = 5  # keep small — 25 samples at 400 DPI caused OOM on Kaggle
        indices = random.sample(range(len(dataset)), k) if len(dataset) >= k else list(range(len(dataset)))

    n = len(indices)
    fig, axs = plt.subplots(n, 2, figsize=(8, 4 * n))
    if n == 1:
        axs = axs[np.newaxis]

    with torch.no_grad():
        for row, idx in enumerate(indices):
            x_t, y_t, _ = dataset[idx]
            y_pred = model(x_t.unsqueeze(0).to(device))

            margin = 64
            _, _, H, W = y_pred.shape
            y_pred = y_pred[:, :, margin:H - margin, margin:W - margin]
            y_t    = y_t.unsqueeze(0).to(device)[:, :, margin:H - margin, margin:W - margin]

            gt = y_t.squeeze().cpu().numpy().astype(np.float32)
            pr = y_pred.squeeze().cpu().numpy().astype(np.float32)
            if gt.ndim == 3: gt = gt[target_channel]
            if pr.ndim == 3: pr = pr[target_channel]

            # Rescale prediction to GT statistics (visualization only — hides quantitative errors)
            pr_adj = (pr - pr.mean()) / (pr.std() + eps) * gt.std() + gt.mean()
            s_min  = min(gt.min(), pr_adj.min())
            s_max  = max(gt.max(), pr_adj.max()) + eps

            axs[row, 0].imshow((gt - s_min) / (s_max - s_min),     cmap="inferno"); axs[row, 0].set_title(f"GT (idx {idx})"); axs[row, 0].axis("off")
            axs[row, 1].imshow((pr_adj - s_min) / (s_max - s_min), cmap="inferno"); axs[row, 1].set_title("Prediction");      axs[row, 1].axis("off")

    fig.suptitle(name, fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    out_path = f"comparison_grid_{name}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")  # 150 DPI sufficient; 400 caused OOM
    plt.show(); plt.close(fig)
    print(f"[vis] Saved to: {out_path}")


def smart_load_state_dict(model, state, strict=True):
    is_dp      = isinstance(model, torch.nn.DataParallel)
    has_module = all(k.startswith("module.") for k in state.keys())
    if is_dp and not has_module:
        # Checkpoint from non-DP run, loading into DP model → load into inner module
        return model.module.load_state_dict(state, strict=strict)
    if not is_dp and has_module:
        # Checkpoint from DP run, loading into non-DP model → strip module. prefix
        state = {k.replace("module.", "", 1): v for k, v in state.items()}
    # Remaining cases: (DP+DP) or (non-DP+non-DP) → load into model directly
    return model.load_state_dict(state, strict=strict)


def train_model_with_config(model, train_loader, val_loader, device, title, config, save_path=None):
    """Config-based training used by Optuna CV. Returns best validation loss."""
    model.to(device)
    lr                = config.get('lr', 1e-4)
    alpha             = config.get('alpha', 0.8)
    huber_beta        = config.get('huber_beta', 1.0)
    weight_decay      = config.get('weight_decay', 1e-5)
    grad_clip         = config.get('grad_clip', 1.0)
    scheduler_patience = config.get('scheduler_patience', 3)
    scheduler_factor  = config.get('scheduler_factor', 0.7)
    epochs            = config.get('epochs', 50)
    early_stop_patience = config.get('early_stop_patience', 10)
    epsilon           = 1e-8

    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=scheduler_factor, patience=scheduler_patience)
    scaler = GradScaler(enabled=True)
    best_val, no_improve = float("inf"), 0

    for epoch in range(epochs):
        model.train()
        running = 0.0
        for x, y, original_sizes in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            B = x.shape[0]
            opt.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', enabled=True):
                yhat = model(x)
                hub_sum = ssim_sum = 0.0
                for i in range(B):
                    h, w = original_sizes[i]
                    y_i, yhat_i = y[i:i+1, :, :h, :w], yhat[i:i+1, :, :h, :w]
                    y_n    = (y_i - y_i.min().detach()) / (y_i.max().detach() - y_i.min().detach() + epsilon)
                    yhat_n = (yhat_i - yhat_i.min().detach()) / (yhat_i.max().detach() - yhat_i.min().detach() + epsilon)
                    hub_sum  += F.smooth_l1_loss(yhat_i, y_i, beta=huber_beta, reduction="mean")
                    ssim_sum += 1.0 - ms_ssim(yhat_n, y_n, data_range=1.0, win_size=7, weights=[0.4, 0.3, 0.3])
                loss = alpha * torch.as_tensor(hub_sum / B, device=device) \
                     + (1 - alpha) * torch.as_tensor(ssim_sum / B, device=device)
            scaler.scale(loss).backward()
            if grad_clip:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(opt); scaler.update()
            running += loss.item() * B

        train_loss = running / len(train_loader.dataset)
        model.eval()
        val_running, valid_batches = 0.0, 0
        with torch.no_grad(), autocast(device_type='cuda', enabled=True):
            for x, y, original_sizes in val_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                B = x.shape[0]; yhat = model(x)
                hub_sum = ssim_sum = 0.0; skip = False
                for i in range(B):
                    h, w   = original_sizes[i]
                    y_i    = y[i:i+1, :, :h, :w]
                    yhat_i = yhat[i:i+1, :, :h, :w].clamp(0.0, 1.0)
                    y_n    = (y_i - y_i.min()) / ((y_i.max() - y_i.min()).abs() + epsilon)
                    yhat_n = (yhat_i - yhat_i.min()) / ((yhat_i.max() - yhat_i.min()).abs() + epsilon)
                    ssim_val = ms_ssim(yhat_n, y_n, data_range=1.0, win_size=7, weights=[0.4, 0.3, 0.3])
                    hub_val  = F.smooth_l1_loss(yhat_i, y_i, beta=huber_beta, reduction="mean")
                    if torch.isnan(ssim_val) or torch.isnan(hub_val): skip = True; break
                    hub_sum += hub_val; ssim_sum += 1.0 - ssim_val
                if not skip:
                    vl = (alpha * hub_sum + (1 - alpha) * ssim_sum) / B
                    if not (torch.isnan(vl) or torch.isinf(vl)):
                        val_running += float(vl) * B; valid_batches += B

        val_loss = val_running / valid_batches if valid_batches > 0 else float('nan')
        sched.step(val_loss)
        if val_loss + 1e-6 < best_val:
            best_val = val_loss; no_improve = 0
            if save_path: torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1
            if no_improve >= early_stop_patience:
                print(f"Early stop at epoch {epoch+1}, best val={best_val:.4f}"); break
        print(f"Epoch {epoch+1}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    return best_val

In [ ]:
def main(
    path: str,
    target_idx: int,
    control_markers_indices: list,
    shared_marker_indices: list = None,
    title: str = "Unspecified",
    n_splits: int = 3,
    batch_size: int = 2,
    device: str = None,
    data_name: str = None,
    cache_dir: str = "./cv_cache",
    resume: bool = True,
    n_components: int = 3,
    cofactor: float = 5.0,
    n_trials: int = 10,
    finetune: bool = False,
    test_path: str = None,
    test_target_idx: int = None,
    test_index_map: dict = None,
    test_control_markers_indices: list = None,
    use_pca: bool = True,
    pretrained_conv1: bool = False,
):
    '''
    Complete workflow:
    1. Split data into train (75%) and test (25%), or use external test_path
    2. Optionally run hyperparameter optimization with k-fold CV on train set (finetune=True)
    3. Train final model on full train set with best hyperparameters
    4. Evaluate on held-out test set

    use_pca=True  → regression + arcsinh + PCA(n_components) + z-score (baseline)
    use_pca=False → regression + arcsinh + per-channel z-score (no PCA bottleneck)
    pretrained_conv1=True → use backbone.conv1 weights (only valid when use_pca=True, in_channels=3)
    '''

    os.makedirs(cache_dir, exist_ok=True)
    set_seed(42)

    best_hyperparams = {'lr': 1e-04, 'alpha': 0.985}

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # Model cache depends on both flags; preprocessing only depends on use_pca,
    # so baseline and pretrained_conv1 share the same fitted preprocessor.
    tag         = f"pca{use_pca}_ptconv{pretrained_conv1}"
    preproc_tag = f"pca{use_pca}"

    all_files = sorted([
        os.path.join(path, f) for f in os.listdir(path) 
        if f.endswith(('.tif', '.tiff'))
    ])

    if test_path is not None:
        train_files = all_files
        test_files = sorted(
            os.path.join(test_path, f)
            for f in os.listdir(test_path)
            if f.endswith(('.tif', '.tiff'))
        )
        print(f"[split] Train: {len(train_files)}, External test: {len(test_files)} from {test_path}")
    else:
        split_path = os.path.join(cache_dir, f"train_test_split_{data_name}_{title}.pkl")
        if resume and os.path.exists(split_path):
            with open(split_path, "rb") as f:
                train_files, test_files = pickle.load(f)
            print(f"[init] Loaded train/test split from {split_path}")
        else:
            random.shuffle(all_files)
            split_idx = int(0.75 * len(all_files))
            train_files = all_files[:split_idx]
            test_files = all_files[split_idx:]
            with open(split_path, "wb") as f:
                pickle.dump((train_files, test_files), f)
            print(f"[init] Created train/test split: {len(train_files)} train, {len(test_files)} test")

        print(f"[split] Train: {len(train_files)}, Test: {len(test_files)}")
    
    if finetune:
        cv_files = random.sample(train_files, len(train_files)//3)

        best_hyperparams = run_cv_optimization(
            train_files=cv_files,
            target_idx=target_idx,
            control_markers_indices=control_markers_indices,
            title=title,
            data_name=data_name,
            n_splits=n_splits,
            batch_size=batch_size,
            device=device,
            cache_dir=cache_dir,
            n_components=n_components,
            cofactor=cofactor,
            n_trials=n_trials,
            resume=resume
        )

        hyperparam_path = os.path.join(cache_dir, f"best_hyperparams_{data_name}_{title}_{tag}.json")
        with open(hyperparam_path, "w") as f:
            json.dump(best_hyperparams, f, indent=2)
        print(f"[cache] Saved best hyperparameters to {hyperparam_path}")

    final_model_path = os.path.join(cache_dir, f"final_model_{data_name}_{title}_{tag}.pth")
    preproc_path     = os.path.join(cache_dir, f"final_preproc_{data_name}_{title}_{preproc_tag}.pkl")

    # Fit or load preprocessing
    if resume and os.path.exists(preproc_path):
        with open(preproc_path, "rb") as f:
            reg_models, pca, pca_mean, pca_std, input_channels = pickle.load(f)
        print(f"[init] Loaded preprocessing from {preproc_path}")
    else:
        print(f"[prepro] Fitting on {len(train_files)} training images (use_pca={use_pca})...")
        reg_models, pca, pca_mean, pca_std, input_channels = fit_preprocessing(
            train_files, target_idx, control_markers_indices,
            shared_marker_indices=shared_marker_indices,
            n_components=n_components, cofactor=cofactor,
            use_pca=use_pca,
        )
        with open(preproc_path, "wb") as f:
            pickle.dump((reg_models, pca, pca_mean, pca_std, input_channels), f)
        print(f"[cache] Saved preprocessing to {preproc_path}")

    # Determine model input channels based on preprocessing mode
    in_channels = n_components if use_pca else len(input_channels)
    print(f"[model] in_channels={in_channels}, pretrained_conv1={pretrained_conv1}")

    # Create datasets
    train_dataset = CustomDataset(
        train_files, target_idx, control_markers_indices,
        reg_models, pca, pca_mean, pca_std, input_channels,
        n_components=n_components, cofactor=cofactor
    )

    tt_idx   = test_target_idx if test_target_idx is not None else target_idx
    test_ctrl = test_control_markers_indices if test_control_markers_indices is not None else control_markers_indices

    test_dataset = CustomDataset(
        test_files, tt_idx, test_ctrl,
        reg_models, pca, pca_mean, pca_std, input_channels,
        n_components=n_components, cofactor=cofactor,
        index_map=test_index_map,
    )
    
    train_size = int(0.75 * len(train_dataset))
    val_size   = len(train_dataset) - train_size

    train_subset, val_subset = random_split(
        train_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    # num_workers=0: avoids spawning worker processes that each pin a full TIFF in RAM
    train_loader = DataLoader(
        train_subset, batch_size=batch_size, shuffle=True,
        collate_fn=pad_collate, num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        val_subset, batch_size=batch_size, shuffle=False,
        collate_fn=pad_collate, num_workers=0, pin_memory=True
    )

    # Build model
    model = ResNetUNet(in_channels=in_channels, pretrained_conv1=pretrained_conv1).to(device)
    
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)

    if resume and os.path.exists(final_model_path):
        print(f"[init] Loading existing final model from {final_model_path}")
        state = torch.load(final_model_path, map_location=device)
        smart_load_state_dict(model, state, strict=True)
    else:
        print(f"[training] Training final model")
        # epochs=30, patience=8: observed convergence at ~epoch 22, LR cooldown benefit by epoch 30
        train_model(
            model, train_loader, val_loader, device,
            config=best_hyperparams,
            title=f"{title} ({tag})",
            epochs=30,
            patience=8,
            save_path=final_model_path,
            dataset=data_name
        )
        print(f"[cache] Saved final model to {final_model_path}")

    print(f"\n{'='*10}")
    print(f"FINAL EVALUATION  [{tag}]")
    print(f"{'='*10}")

    test_metrics = evaluate_model(
        model=model,
        dataset=test_dataset,
        device=device,
        batch_size=batch_size,
        title=title,
        data_name=data_name
    )

    print(f"[FINAL RESULTS] {title}  [{tag}]")
    print(f"  Test RMSE:    {test_metrics['rmse']:.4f}")
    print(f"  Test Pearson: {test_metrics['pearson']:.4f}")
    print(f"  Test FID:     {test_metrics['fid']:.2f}")
    print(f"  Hyperparams:  {best_hyperparams}")

    visualize_prediction(
        model=model,
        dataset=test_dataset,
        device=device,
        indices=None,
        target_channel=0,
        name=f"{title}_{tag}"
    )

    metrics_path = os.path.join(cache_dir, f"final_metrics_{data_name}_{title}_{tag}.json")
    with open(metrics_path, "w") as f:
        json.dump(test_metrics, f, indent=2)
    print(f"[cache] Saved final metrics to {metrics_path}")

    return test_metrics

def run_cv_optimization(
    train_files,
    target_idx,
    control_markers_indices,
    title,
    data_name,
    n_splits=3,
    batch_size=2,
    device="cuda",
    cache_dir="./cv_cache",
    n_components=3,
    cofactor=5.0,
    n_trials=10,
    resume=True
):
    '''
    Run k-fold cross-validation with Optuna hyperparameter optimization.
    Uses ONLY train_files - no test set contamination.

    Returns:
        best_hyperparams: dict with optimal 'lr' and 'alpha'
    '''

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_splits = []
    for tr_idx, va_idx in kf.split(train_files):
        tr_files = [train_files[i] for i in tr_idx]
        va_files = [train_files[i] for i in va_idx]
        cv_splits.append((tr_files, va_files))

    print(f"[cv] Created {n_splits}-fold CV splits on {len(train_files)} training files")

    def objective(trial: Trial):
        lr    = trial.suggest_float('lr', 3e-5, 5e-4, log=True)
        alpha = trial.suggest_float('alpha', 0.9, 0.99)

        fold_val_losses = []

        for fold_idx, (tr_files, va_files) in enumerate(cv_splits, 1):
            print(f"[Trial {trial.number}] Fold {fold_idx}/{n_splits} | lr={lr:.2e}, alpha={alpha:.3f}")

            reg_models, pca, pca_mean, pca_std, input_channels = fit_preprocessing(
                tr_files, target_idx, control_markers_indices,
                n_components=n_components, cofactor=cofactor
            )

            tr_dataset = CustomDataset(
                tr_files, target_idx, control_markers_indices,
                reg_models, pca, pca_mean, pca_std, input_channels,
                n_components=n_components, cofactor=cofactor
            )
            va_dataset = CustomDataset(
                va_files, target_idx, control_markers_indices,
                reg_models, pca, pca_mean, pca_std, input_channels,
                n_components=n_components, cofactor=cofactor
            )

            tr_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True,
                                   collate_fn=pad_collate, num_workers=0)
            va_loader = DataLoader(va_dataset, batch_size=batch_size, shuffle=False,
                                   collate_fn=pad_collate, num_workers=0)

            model = ResNetUNet(in_channels=n_components).to(device)
            if torch.cuda.device_count() > 1:
                model = torch.nn.DataParallel(model)

            config = {'lr': lr, 'alpha': alpha, 'epochs': n_trials}  # n_trials epochs for speed

            try:
                val_loss = train_model_with_config(
                    model, tr_loader, va_loader, device, title,
                    config=config,
                    save_path=None
                )
                fold_val_losses.append(val_loss)
            except Exception as e:
                print(f"  [Trial {trial.number}] Fold {fold_idx} failed: {e}")
                fold_val_losses.append(float('inf'))

            del model, tr_dataset, va_dataset, tr_loader, va_loader
            torch.cuda.empty_cache()
            gc.collect()

        mean_val_loss = np.mean(fold_val_losses)
        print(f"  [Trial {trial.number}] Mean CV loss: {mean_val_loss:.4f}")
        return mean_val_loss

    study = optuna.create_study(
        direction='minimize',
        study_name=f"{data_name}_{title}_cv_opt",
        sampler=optuna.samplers.TPESampler(seed=42)
    )

    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    print(f"\n{'='*10}")
    print(f"CV OPTIMIZATION COMPLETE")
    print(f"{'='*10}")
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best CV loss: {study.best_value:.6f}")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")

    return study.best_params

def evaluate_model(model, dataset, device, batch_size, title, data_name):
    '''
    Evaluate model on a dataset and compute RMSE, Pearson, and FID metrics.
    '''
    model.eval()

    test_loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        collate_fn=pad_collate, num_workers=0, pin_memory=True
    )

    sse = 0.0
    npx = 0
    all_ypred_pixels, all_ytrue_pixels = [], []

    with tempfile.TemporaryDirectory() as pred_dir, tempfile.TemporaryDirectory() as true_dir:
        idx_img = 0

        with torch.no_grad(), autocast(device_type='cuda', enabled=True):
            for x, y, original_sizes in test_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                yhat = model(x)

                for b, (h, w) in enumerate(original_sizes):
                    gt = y[b:b+1, :, :h, :w]
                    pr = yhat[b:b+1, :, :h, :w].clamp(0, 1)

                    diff = (pr - gt).float()
                    sse += (diff * diff).sum().item()
                    npx += diff.numel()

                    all_ytrue_pixels.append(gt.detach().cpu().numpy().ravel())
                    all_ypred_pixels.append(pr.detach().cpu().numpy().ravel())

                    gt3 = gt.repeat(1, 3, 1, 1).float()
                    pr3 = pr.repeat(1, 3, 1, 1).float()

                    gt0 = gt3[0]
                    pr0 = pr3[0]
                    gt0 = (gt0 - gt0.min()) / (gt0.max() - gt0.min() + 1e-8)
                    pr0 = (pr0 - pr0.min()) / (pr0.max() - pr0.min() + 1e-8)

                    gt_np = gt0.permute(1, 2, 0).cpu().numpy()
                    pr_np = pr0.permute(1, 2, 0).cpu().numpy()

                    gt_np = skimage.transform.resize(gt_np, (512, 512, 3), anti_aliasing=True, mode='reflect')
                    pr_np = skimage.transform.resize(pr_np, (512, 512, 3), anti_aliasing=True, mode='reflect')

                    imageio.imwrite(os.path.join(true_dir, f"{idx_img}.png"), (gt_np * 255).astype(np.uint8))
                    imageio.imwrite(os.path.join(pred_dir, f"{idx_img}.png"), (pr_np * 255).astype(np.uint8))
                    idx_img += 1

        fid_val = fid.compute_fid(pred_dir, true_dir, mode="clean", verbose=False)

    ypred = np.concatenate(all_ypred_pixels)
    ytrue = np.concatenate(all_ytrue_pixels)
    pearson_val, _ = pearsonr(ypred, ytrue)
    rmse = float(np.sqrt(sse / max(npx, 1)))

    return {
        'marker': title,
        'rmse': rmse,
        'pearson': float(pearson_val),
        'fid': float(fid_val)
    }

In [7]:
# Cross-dataset leave-one-marker-out:
# Train on jackson shared markers only, test on danenberg using a channel-index map.

train_dataset = "jackson2020"
train_path = f"/kaggle/input/{train_dataset}"

test_dataset = "danenberg2022"
test_path = f"/kaggle/input/{test_dataset}"

# Marker name lists (one per channel, in the exact order used in each dataset's TIFFs)
d_markers = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163', 'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18', 'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20', 'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4', 'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
j_markers = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18', 'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho', 'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20', 'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7', 'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
# not sure regarding Jackson ERa and HER2

# 1) Shared marker names
shared_markers = sorted(set(j_markers).intersection(set(d_markers)))
print("Shared markers (n=%d):" % len(shared_markers))
print(shared_markers)

# 2) Name -> channel index maps
j_idx = {m:i for i,m in enumerate(j_markers)}
d_idx = {m:i for i,m in enumerate(d_markers)}

# 3) Train-index -> test-index map (so we can reuse the jackson-trained preprocessing on danenberg)
train_to_test_index_map = { j_idx[m]: d_idx[m] for m in shared_markers }

# 4) Restrict training inputs to shared markers (indices in jackson channel space)
shared_train_indices = [j_idx[m] for m in shared_markers]

# 5) Controls (choose controls by *name* so the order is consistent across datasets)
#    IMPORTANT: These controls must exist in both datasets.
control_names = ["histone_h3", 'dna1', 'dna2']  
control_train = [j_idx[m] for m in control_names]

# 6) Leave-one-out tasks: only predict markers that are shared (and not in the control set).
exclude_from_prediction = set(control_names) 
marker_tasks = [(m, j_idx[m]) for m in shared_markers if m not in exclude_from_prediction]

Shared markers (n=14):
['c_erb_b_2_her2', 'cd20', 'cd3', 'cd45', 'cd68', 'cytokeratin_5', 'cytokeratin_8_18', 'dna1', 'dna2', 'estrogen_receptor_alpha', 'histone_h3', 'ki_67', 'pan_cytokeratin', 'sma']


In [ ]:
# Three-way ablation:
#   baseline         — PCA (3D) + random conv1          (current approach, correctly implemented)
#   no_pca           — per-channel z-score + random conv1  (removes PCA bottleneck)
#   pretrained_conv1 — PCA (3D) + pretrained conv1       (actually uses ImageNet weights)

# Limit to 3 representative markers to keep the run within ~9 hours on Kaggle GPU.
# Full 11-marker run would take ~100 hours (4 min/epoch × 50 epochs × 11 × 3 conditions).
ablation_marker_tasks = [
    ('c_erb_b_2_her2',        j_idx['c_erb_b_2_her2']),        # proliferative / HER2+
    ('cytokeratin_5',         j_idx['cytokeratin_5']),          # basal epithelial
    ('estrogen_receptor_alpha', j_idx['estrogen_receptor_alpha']),  # luminal / ER+
]

conditions = [
    {"label": "baseline",           "use_pca": True,  "pretrained_conv1": False},
    {"label": "no_pca",             "use_pca": False, "pretrained_conv1": False},
    {"label": "pretrained_conv1",   "use_pca": True,  "pretrained_conv1": True},
]

all_results = []

for cond in conditions:
    print(f"\n{'='*20}")
    print(f"CONDITION: {cond['label']}")
    print(f"{'='*20}")

    for idx, (title, train_target_idx) in enumerate(ablation_marker_tasks):

        print(f"\n  Marker {idx+1}/{len(ablation_marker_tasks)}: {title}")
        start_time = time.time()

        result = main(
            path=train_path,
            target_idx=train_target_idx,
            control_markers_indices=control_train,
            shared_marker_indices=shared_train_indices,
            title=title,
            data_name=f"{train_dataset}_to_{test_dataset}_{cond['label']}",
            cache_dir=WORK_CACHE_ROOT,
            n_splits=3,
            batch_size=2,
            n_components=3,
            cofactor=5.0,
            finetune=False,
            test_path=test_path,
            test_index_map=train_to_test_index_map,
            use_pca=cond["use_pca"],
            pretrained_conv1=cond["pretrained_conv1"],
        )

        elapsed = (time.time() - start_time) / 60
        print(f"  Finished {title} [{cond['label']}] in {elapsed:.1f} min")

        all_results.append({
            "condition":  cond["label"],
            "marker":     title,
            "rmse":       result["rmse"],
            "pearson":    result["pearson"],
            "fid":        result["fid"],
        })

        # Free GPU memory between runs to prevent accumulation across markers
        torch.cuda.empty_cache()
        gc.collect()

# Summary table
results_df = pd.DataFrame(all_results)

print("\n\n" + "="*50)
print("ABLATION SUMMARY")
print("="*50)

for metric in ["rmse", "pearson", "fid"]:
    print(f"\n--- {metric.upper()} ---")
    pivot = results_df.pivot_table(index="marker", columns="condition", values=metric)
    ordered_cols = [c["label"] for c in conditions if c["label"] in pivot.columns]
    print(pivot[ordered_cols].round(4).to_string())

results_df.to_csv(os.path.join(WORK_CACHE_ROOT, "ablation_results.csv"), index=False)
print(f"\n[saved] ablation_results.csv")